In [22]:
#Perform your data wrangling and pre-processing (handle missing values, encode categoricals, scale as needed)

#loading needed packages
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [23]:
#Loading and previewing the Dataset
data = pd.read_csv("../Data/facebook+metrics/dataset_Facebook.csv", sep=";")
data.head()

,Page total likes,Type,Category,Post Month,Post Weekday,Post Hour,Paid,Lifetime Post Total Reach,Lifetime Post Total Impressions,Lifetime Engaged Users,Lifetime Post Consumers,Lifetime Post Consumptions,Lifetime Post Impressions by people who have liked your Page,Lifetime Post reach by people who like your Page,Lifetime People who have liked your Page and engaged with your post,comment,like,share,Total Interactions
0,139441,Photo,2,12,4,3,0.0,2752,5091,178,109,159,3078,1640,119,4,79.0,17.0,100
1,139441,Status,2,12,3,10,0.0,10460,19057,1457,1361,1674,11710,6112,1108,5,130.0,29.0,164
2,139441,Photo,3,12,3,3,0.0,2413,4373,177,113,154,2812,1503,132,0,66.0,14.0,80
3,139441,Photo,2,12,2,10,1.0,50128,87991,2211,790,1119,61027,32048,1386,58,1572.0,147.0,1777
4,139441,Photo,2,12,2,3,0.0,7244,13594,671,410,580,6228,3200,396,19,325.0,49.0,393


In [24]:
#Inspecting Missing Values (missing values is also apart of our EDA)
missing_counts = data.isnull().sum()
missing_counts
#6 missing values out of 17 columns of values is relatively low

Page total likes                                                       0
Type                                                                   0
Category                                                               0
Post Month                                                             0
Post Weekday                                                           0
Post Hour                                                              0
Paid                                                                   1
Lifetime Post Total Reach                                              0
Lifetime Post Total Impressions                                        0
Lifetime Engaged Users                                                 0
Lifetime Post Consumers                                                0
Lifetime Post Consumptions                                             0
Lifetime Post Impressions by people who have liked your Page           0
Lifetime Post reach by people who like your Page   

In [25]:
#Here we handle the missing values by:
## Filling  missing 'numeric engagement methods' with 0
## Filling missing 'reach/impression' metrics with median
## Filling missing categorical values with mode

# Numeric columns to fill with 0
zero_fill_cols = ['comment', 'like', 'share', 'Lifetime Engaged Users']

for col in zero_fill_cols:
    data[col] = data[col].fillna(0)

# Numeric columns to fill with median
median_fill_cols = [
    'Lifetime Post Total Reach', 'Lifetime Post Total Impressions',
    'Lifetime Post Consumers', 'Lifetime Post Consumptions'
]

for col in median_fill_cols:
    data[col] = data[col].fillna(data[col].median())

# Categorical columns to fill with mode
categorical_cols = ['Type', 'Category', 'Post Month', 'Post Weekday', 'Post Hour', 'Paid']

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

In [26]:
#Inspecting if there are any duplicates
duplicate_count = data.duplicated().sum()
duplicate_count

#our data has no duplicate rows 

np.int64(0)

In [27]:
#Based on our Insights from our EDA, "Page Total Likes" rarely correlates with engagement. Columns with extremely high multicollinearity (columns that are telling the model the same thing) can be removed. Columns with >80% missing values should be removed. 

drop_cols = ['Page total likes', 
             'Lifetime Post Total Impressions',
            'Lifetime Post Consumptions',
            'Total Interactions'
            ]  

data = data.drop(columns=drop_cols)

# We are choosing to drop the columns: 'Page Total Likes' , 'Lifetime Post Total Impressions','Lifetime Post Consumptions', and 'Total Interactions'.

                                     Why we chose to drop columns: 

We removed four columns from the dataset because they were either not useful or too similar to other columns and can cause redundancy and confusion with our model. Thus, making our model less accurate. 

Our goal is to: 

- Remove redundant information

- Reduce multicollinearity

- Make the model simpler and more stable

- Improve predictive performance

**1. Page total likes** : had almost no relationship with engagement (likes, comments, shares). Since it doesn’t help predict anything, we dropped it. 

**2. Lifetime Post Total Impressions** : This metric was almost identical to Lifetime Post Total Reach which measures how many people saw the post (correlation ~ 0.99). WE kept "reach" and dropped "impressions". Keeping both creates multicollinearity — the model seeing the same information twice.

**3. Lifetime Post Consumptions**: correlated with Lifetime post Consumers (correlation~ 0.99). 
Comsumptions = total clicks

Consumer = Unique users who clicked

Therefore, we kept consumers which is more meaningful and dropped consumptions.

**4. Total Interactions**: The sum of like, comment, and share. Keeping it would duplicate the same engament signal becuase it has high corellation with all 3 features. we decided to keep the individual metrics to better clean the data. 


In [28]:
#Encode Categorical Values - so our ML model can translate/ understand the text in terms that make sense for the model, numerical values.
data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=True)
data_encoded.head()

,Lifetime Post Total Reach,Lifetime Engaged Users,Lifetime Post Consumers,Lifetime Post Impressions by people who have liked your Page,Lifetime Post reach by people who like your Page,Lifetime People who have liked your Page and engaged with your post,comment,like,share,Type_Photo,...,Post Hour_14,Post Hour_15,Post Hour_16,Post Hour_17,Post Hour_18,Post Hour_19,Post Hour_20,Post Hour_22,Post Hour_23,Paid_1.0
0,2752,178,109,3078,1640,119,4,79.0,17.0,True,...,False,False,False,False,False,False,False,False,False,False
1,10460,1457,1361,11710,6112,1108,5,130.0,29.0,False,...,False,False,False,False,False,False,False,False,False,False
2,2413,177,113,2812,1503,132,0,66.0,14.0,True,...,False,False,False,False,False,False,False,False,False,False
3,50128,2211,790,61027,32048,1386,58,1572.0,147.0,True,...,False,False,False,False,False,False,False,False,False,True
4,7244,671,410,6228,3200,396,19,325.0,49.0,True,...,False,False,False,False,False,False,False,False,False,False


In [29]:
#Scale Numerical Columns - give our numeric columns a similar range,  similar to standardizing.  
num_cols = [
    'Lifetime Post Total Reach', 'Lifetime Engaged Users', 
    'comment', 'like', 'share'
]

scaler = StandardScaler()
data_encoded[num_cols] = scaler.fit_transform(data_encoded[num_cols])

In [30]:
#Finally Save the Clean Dataset
data_encoded.to_csv("../processed/facebook_cleaned.csv", index=False)

In [31]:
#Train/ Test Split 
#We chose 'like' because 'like' is the most stable/ computable metric to evaluate. 
X = data_encoded.drop(columns=['like'])
y = data_encoded['like']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)